In [27]:
pip install deep_translator

In [28]:
import pandas as pd
import re
import time
from deep_translator import GoogleTranslator
import nltk
from nltk.corpus import stopwords

In [29]:
# load dataset
df = pd.read_csv("/content/cv scan parse.csv")

# cek data
print(df.head())
print(df.columns)
print(len(df))


   candidate_name                       email           phone  \
0  Siti Nurhaliza  siti.nurhaliza@example.com   6281234567890   
1      Rina Aulia      rina.aulia@example.com   6281234567890   
2    Anna Suryani    anna.suryani@example.com   6281234567890   
3    Andi Pratama    andi.pratama@example.com   6281123456789   
4        John Doe          john.doe@email.com  62898765432100   

                                              skills  \
0  penanganan keluhan, komunikasi efektif, penyel...   
1  komunikasi efektif, crm software, salesforce, ...   
2  problem-solving, intern midwife august 2023 - ...   
3  sistem pos, komunikasi efektif, manajemen inve...   
4  assisted in planning and executing marketing c...   

                                             summary  \
0                                               NONE   
1                                               NONE   
2                                               NONE   
3                                               

In [30]:
df.shape

(366, 12)

In [31]:
df.isna().sum()

,0
candidate_name,0
email,0
phone,0
skills,0
summary,0
experience,0
degree,0
university,0
education,0
text,0


In [32]:
# hapus row yang text-nya kosong
df = df.dropna(subset=["text"])

# isi missing value
df["skills"] = df["skills"].fillna("-")
df["summary"] = df["summary"].fillna("-")
df["experience"] = df["experience"].fillna("-")
df["degree"] = df["degree"].fillna("-")
df["university"] = df["university"].fillna("-")
df["education"] = df["education"].fillna("-")

# # kalau kolom preprocessing ada
# df["translated_text"] = df["translated_text"].fillna("-")
# df["clean_text_aggressive"] = df["clean_text_aggressive"].fillna("-")
# df["clean_text_light"] = df["clean_text_light"].fillna("-")

# cek missing value
df.isna().sum()

,0
candidate_name,0
email,0
phone,0
skills,0
summary,0
experience,0
degree,0
university,0
education,0
text,0


# MINI PREPRO

In [33]:
def clean_text(text):

    text = text.lower()

    # hapus angka
    text = re.sub(r'\d+', ' ', text)

    # hapus simbol
    text = re.sub(r'[^a-z\s]', ' ', text)

    # hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# gabungkan informasi CV
df["text"] = (
    df["summary"].fillna("") + " " +
    df["experience"].fillna("") + " " +
    df["skills"].fillna("") + " " +
    df["education"].fillna("")
)

# clean text dasar
df["text"] = df["text"].apply(clean_text)

df.head()

,candidate_name,email,phone,skills,summary,experience,degree,university,education,text,translated_text,clean_text
0,Siti Nurhaliza,siti.nurhaliza@example.com,6281234567890,"penanganan keluhan, komunikasi efektif, penyel...",NONE,lebih dari 2 tahun dalam menangani pertanyaan ...,SMA,lorem,sma negeri 2 jakarta agustus 2018 - juli 2021 ...,none lebih dari tahun dalam menangani pertanya...,Summary: Experience: more than 2 years in effi...,summary experience more than 2 years in effici...
1,Rina Aulia,rina.aulia@example.com,6281234567890,"komunikasi efektif, crm software, salesforce, ...",NONE,3 tahun dalam memberikan pelayanan yang optima...,D3,universitas lorem ipsum jakarta lulus dengan p...,d3 administrasi bisnis september 2018 - juli 2...,none tahun dalam memberikan pelayanan yang opt...,Summary: Experience: 3 years in providing opti...,summary experience 3 years in providing optima...
2,Anna Suryani,anna.suryani@example.com,6281234567890,"problem-solving, intern midwife august 2023 - ...",NONE,d in providing comprehensive care to expectant...,S1,university of indonesia jakarta focused on mat...,", and teamwork within a healthcare environment...",none d in providing comprehensive care to expe...,Summary: Experience: d in providing comprehens...,summary experience d in providing comprehensiv...
3,Andi Pratama,andi.pratama@example.com,6281123456789,"sistem pos, komunikasi efektif, manajemen inve...",NONE,lebih dari 3 tahun di industri kopi. terampil ...,SMA,lorem,sma negeri 3 jakarta agustus 2017 - juli 2020 ...,none lebih dari tahun di industri kopi terampi...,Summary: Experience: more than 3 years in the ...,summary experience more than 3 years in the co...
4,John Doe,john.doe@email.com,62898765432100,assisted in planning and executing marketing c...,a recent high school graduate with a strong ac...,internship - marketing assistant august 2022 -...,D3,lorem,high school diploma august 2020 - july 2023 lo...,a recent high school graduate with a strong ac...,Summary: a recent high school graduate with a ...,summary a recent high school graduate with a s...


# BALANCING

In [34]:
categories = {

    "Data": [
        "data analyst", "data analytics", "data scientist", "data science",
        "machine learning", "ml engineer", "deep learning", "ai engineer",
        "data engineer", "big data", "data mining", "data visualization",
        "analisis data", "ilmuwan data", "pengolahan data","data"
    ],

    "Software": [
        "software engineer", "software developer", "developer", "programmer",
        "backend", "backend developer", "frontend", "frontend developer",
        "full stack", "fullstack", "web developer", "mobile developer","web","api","mobile",
        "android developer", "ios developer", "pengembang", "coding", "komputer","computer"
    ],

    "Marketing": [
        "marketing", "digital marketing", "seo", "sem", "content marketing",
        "branding", "social media", "campaign", "market research",
        "pemasaran", "iklan", "advertising"
    ],

    "Finance": [
        "finance", "financial", "accounting", "accountant", "tax", "auditor",
        "budgeting", "investment", "banking", "keuangan", "akuntansi",
        "pajak", "audit","penagihan","collection"
    ],

    "HR": [
        "hr", "human resource", "recruitment", "recruiter",
        "talent acquisition", "people development", "training",
        "sumber daya manusia", "rekrutmen", "hrd"
    ],

    "Design & Creative": [
        "ui", "ux", "ui ux", "ui/ux", "product design",
        "graphic design", "graphic designer", "visual design",
        "creative design", "desain grafis", "desainer",
        "videographer", "photographer", "editor","kreatif",
        "content creator", "tiktok", "media","design",
        "creative", "social media content","live"
    ],

    "Operations": [
        "operations", "operational", "operation staff","inventory","stock","display",
        "logistics", "supply chain", "warehouse", "inventory",
        "procurement", "purchasing", "gudang","supervisor","k3","keselamatan kerja"
    ],

    "Sales": [
        "sales", "sales executive", "account manager",
        "business development", "bd", "client relation",
        "penjualan", "sales marketing", "consultant","account executive"
    ],

    "Engineering": [
        "engineer", "engineering", "mechanical", "electrical","prototipe","architect","arsitek",
        "civil", "industrial engineer", "teknik mesin","proses produksi","drilling","teknik industri",
        "teknik sipil", "teknik elektro","elektro","listrik","mekanik","alat berat"
    ],

    "Customer Service": [
        "customer service", "customer support", "support",
        "call center", "helpdesk", "cs", "layanan pelanggan","service"
    ],
    "Admin": [
        "admin", "administration", "administrasi", "asisten",
        "secretary", "sekretaris", "dokumen","document",
        "assistant", "personal assistant","document controller","legal"
    ],

    "Hospitality": [
        "hotel", "waiter", "waitress", "barista", "chef", "supir","sopir","pelayan","pelayanan",
        "kitchen", "restaurant", "f&b", "food", "beverage","masak","memasak",
        "front office", "guest service", "housekeeping","baker","driver","pembantu","tamu"
    ],

    "Healthcare": [
        "perawat", "nurse", "kesehatan", "medical","medis","gizi","ahli gizi","mental","psikiater","psikolog",
        "lab", "bioteknologi", "farmasi","medic","healthcare","dokter","apotek","safety"
    ],

    "Education": [
        "guru", "teacher", "school", "walikelas", "mengajar","pengajar","pelajaran", "les",
        "dosen", "education", "anak", "pendidikan", "pengajar","siswa","kelas","sekolah"
    ],

    "Other": []
}


In [35]:
def classify_job(text):
    scores = {}

    for category, keywords in categories.items():
        score = 0
        for keyword in keywords:
            if keyword in text:
                # keyword panjang lebih kuat
                score += 2 if len(keyword.split()) > 1 else 1
        scores[category] = score

    best_category = max(scores, key=scores.get)

    if scores[best_category] == 0:
        return "Other"

    return best_category

In [36]:
df["Category"] = df["text"].apply(classify_job)

# lihat hasil
print(df[["candidate_name", "Category"]].head())

   candidate_name          Category
0  Siti Nurhaliza       Hospitality
1      Rina Aulia  Customer Service
2    Anna Suryani        Healthcare
3    Andi Pratama  Customer Service
4        John Doe         Marketing


In [37]:
print(df["Category"].value_counts())


Category
Data                 48
Engineering          47
Healthcare           46
Marketing            37
Design & Creative    32
Customer Service     31
Software             30
Education            23
Hospitality          21
Admin                21
Finance               9
Operations            8
HR                    8
Sales                 3
Other                 2
Name: count, dtype: int64


In [38]:
# 1. buang Other
df_filtered = df[df["Category"] != "Other"].copy()

# 2. fungsi sampling per kategori
def smart_sample(group):

    # kalau data > 100 → ambil 100
    if len(group) > 100:
        return group.sample(100, random_state=42)

    # kalau <= 100 → ambil semua
    return group

# 3. apply per kategori
df_balanced = (
    df_filtered
    .groupby("Category", group_keys=False)
    .apply(smart_sample)
    .reset_index(drop=True)
)

# 4. cek hasil
print(df_balanced["Category"].value_counts())

Category
Data                 48
Engineering          47
Healthcare           46
Marketing            37
Design & Creative    32
Customer Service     31
Software             30
Education            23
Hospitality          21
Admin                21
Finance               9
Operations            8
HR                    8
Sales                 3
Name: count, dtype: int64


/tmp/ipykernel_9731/3460269141.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(smart_sample)


In [39]:
df_balanced.to_csv("dataset_balanced.csv", index=False)

In [40]:
dfb = pd.read_csv("dataset_balanced.csv")

# cek data
print(df.head())
print(df.columns)
print(len(df))

   candidate_name                       email           phone  \
0  Siti Nurhaliza  siti.nurhaliza@example.com   6281234567890   
1      Rina Aulia      rina.aulia@example.com   6281234567890   
2    Anna Suryani    anna.suryani@example.com   6281234567890   
3    Andi Pratama    andi.pratama@example.com   6281123456789   
4        John Doe          john.doe@email.com  62898765432100   

                                              skills  \
0  penanganan keluhan, komunikasi efektif, penyel...   
1  komunikasi efektif, crm software, salesforce, ...   
2  problem-solving, intern midwife august 2023 - ...   
3  sistem pos, komunikasi efektif, manajemen inve...   
4  assisted in planning and executing marketing c...   

                                             summary  \
0                                               NONE   
1                                               NONE   
2                                               NONE   
3                                               

In [41]:
translated_texts = []

def translate_text(text):
    try:
        if text is None or str(text).strip() == "":
            return text

        # Indonesia → English
        return GoogleTranslator(source="auto", target="en").translate(text)

    except Exception as e:
        print(f"Error: {e}")
        return text  # fallback

# proses translate
for i, text in enumerate(dfb["text"]):
    print(f"Translating {i+1}/{len(dfb)}...")

    translated = translate_text(text)
    translated_texts.append(translated)

    time.sleep(0.5)  # antisipasi limit API

print("✅ Translate selesai")

Translating 1/364...
Translating 2/364...
Translating 3/364...
Translating 4/364...
Translating 5/364...
Translating 6/364...
Translating 7/364...
Translating 8/364...
Translating 9/364...
Translating 10/364...
Translating 11/364...
Translating 12/364...
Translating 13/364...
Translating 14/364...
Translating 15/364...
Translating 16/364...
Translating 17/364...
Translating 18/364...
Translating 19/364...
Translating 20/364...
Translating 21/364...
Translating 22/364...
Translating 23/364...
Translating 24/364...
Translating 25/364...
Translating 26/364...
Translating 27/364...
Translating 28/364...
Translating 29/364...
Translating 30/364...
Translating 31/364...
Translating 32/364...
Translating 33/364...
Translating 34/364...
Translating 35/364...
Translating 36/364...
Translating 37/364...
Translating 38/364...
Translating 39/364...
Translating 40/364...
Translating 41/364...
Translating 42/364...
Translating 43/364...
Translating 44/364...
Translating 45/364...
Translating 46/364.

In [42]:
dfb["translated_text"] = translated_texts
dfb[["text", "translated_text"]]

,text,translated_text
0,none lebih dari tahun dalam menangani pertanya...,none more than years in handling customer ques...
1,none tahun dalam memberikan pelayanan yang opt...,none years in providing optimal service to cus...
2,none d in providing comprehensive care to expe...,none d in providing comprehensive care to expe...
3,none lebih dari tahun di industri kopi terampi...,"none more than years in the coffee industry, s..."
4,a recent high school graduate with a strong ac...,a recent high school graduate with a strong ac...
...,...,...
359,none tahun di bidang pelayanan kesehatan di be...,none years in the field of health services in ...
360,profesional it dengan pengalaman lebih dari ta...,IT professional with more than years of experi...
361,none magang di hr department juli september pt...,None Internship in HR Department July Septembe...
362,profesional it dengan pengalaman lebih dari ta...,IT professional with more than years of experi...


In [43]:
# download sekali saja
nltk.download('stopwords')

# load stopwords english
stop_words = set(stopwords.words('english'))

# optional: tambahkan stopwords khusus job (biar lebih clean)
custom_stopwords = {
    "job", "requirement", "requirements", "responsibilities",
    "experience", "skill", "skills", "ability", "abilities",
    "candidate", "position", "role"
}

stop_words = stop_words.union(custom_stopwords)

# fungsi cleaning + stopword removal
def clean_text(text):
    text = str(text).lower()

    # hapus simbol
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    # tokenisasi + hapus stopwords
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]

    return " ".join(tokens)

# apply ke kolom translated_text
dfb["clean_text"] = dfb["translated_text"].apply(clean_text)

# cek hasil
dfb[["translated_text", "clean_text"]].head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,translated_text,clean_text
0,none more than years in handling customer ques...,none years handling customer questions complai...
1,none years in providing optimal service to cus...,none years providing optimal service customers...
2,none d in providing comprehensive care to expe...,none providing comprehensive care expectant mo...
3,"none more than years in the coffee industry, s...",none years coffee industry skilled mixing vari...
4,a recent high school graduate with a strong ac...,recent high school graduate strong academic re...


In [44]:
df_balanced.to_csv("data_clean.csv", index=False)


# EDA

In [45]:
df = pd.read_csv("data_clean.csv")
df

,candidate_name,email,phone,skills,summary,experience,degree,university,education,text,translated_text,clean_text,Category
0,Siti Nurhaliza,siti.nurhaliza@example.com,6281234567890,"penanganan keluhan, komunikasi efektif, penyel...",NONE,lebih dari 2 tahun dalam menangani pertanyaan ...,SMA,lorem,sma negeri 2 jakarta agustus 2018 - juli 2021 ...,none lebih dari tahun dalam menangani pertanya...,Summary: Experience: more than 2 years in effi...,summary experience more than 2 years in effici...,Hospitality
1,Rina Aulia,rina.aulia@example.com,6281234567890,"komunikasi efektif, crm software, salesforce, ...",NONE,3 tahun dalam memberikan pelayanan yang optima...,D3,universitas lorem ipsum jakarta lulus dengan p...,d3 administrasi bisnis september 2018 - juli 2...,none tahun dalam memberikan pelayanan yang opt...,Summary: Experience: 3 years in providing opti...,summary experience 3 years in providing optima...,Customer Service
2,Anna Suryani,anna.suryani@example.com,6281234567890,"problem-solving, intern midwife august 2023 - ...",NONE,d in providing comprehensive care to expectant...,S1,university of indonesia jakarta focused on mat...,", and teamwork within a healthcare environment...",none d in providing comprehensive care to expe...,Summary: Experience: d in providing comprehens...,summary experience d in providing comprehensiv...,Healthcare
3,Andi Pratama,andi.pratama@example.com,6281123456789,"sistem pos, komunikasi efektif, manajemen inve...",NONE,lebih dari 3 tahun di industri kopi. terampil ...,SMA,lorem,sma negeri 3 jakarta agustus 2017 - juli 2020 ...,none lebih dari tahun di industri kopi terampi...,Summary: Experience: more than 3 years in the ...,summary experience more than 3 years in the co...,Customer Service
4,John Doe,john.doe@email.com,62898765432100,assisted in planning and executing marketing c...,a recent high school graduate with a strong ac...,internship - marketing assistant august 2022 -...,D3,lorem,high school diploma august 2020 - july 2023 lo...,a recent high school graduate with a strong ac...,Summary: a recent high school graduate with a ...,summary a recent high school graduate with a s...,Marketing
...,...,...,...,...,...,...,...,...,...,...,...,...,...
359,Nadia Rahma,nadia.rahma@example.com,6281234567890,"pemecahan masalah, prosedur medis minor, empat...",NONE,5 tahun di bidang pelayanan kesehatan di berba...,S1,universitas indonesia jakarta jurusan,sarjana kedokteran september 2014 - agustus 20...,none tahun di bidang pelayanan kesehatan di be...,Summary: Experience: 5 years in the field of h...,summary experience 5 years in the field of hea...,Healthcare
360,Rudi Saputra,rudi.saputra@example.com,6281234567890,kompleks.,profesional it dengan pengalaman lebih dari 3 ...,lebih dari 3 tahun di bidang pengembangan per...,S1,universitas teknologi bandung bandung ipk,sarjana teknik informatika september 2018 - ju...,profesional it dengan pengalaman lebih dari ta...,Summary: IT professional with more than 3 year...,summary it professional with more than 3 years...,Software
361,Dina Pratiwi,dina.pratiwi@loremipsum.com,6281234567890,"komunikasi efektif, riwayat organisasi, pt ips...",NONE,magang di hr department juli 2023 - september ...,lorem,universitas lorem ipsum september 2020,universitas lorem ipsum september 2020 - agust...,none magang di hr department juli september pt...,Summary: Experience: internship at HR departme...,summary experience internship at hr department...,HR
362,Rudi Saputra,rudi.saputra@example.com,6281234567890,kompleks.,profesional it dengan pengalaman lebih dari 3 ...,lebih dari 3 tahun di bidang pengembangan per...,S1,universitas teknologi bandung bandung ipk,sarjana teknik informatika september 2018 - ju...,profesional it dengan pengalaman lebih dari ta...,Summary: IT professional with more than 3 year...,summary it professional with more than 3 years...,Software


## PREPROCESSING UNTUK MODEL EMBENDING

In [47]:
import re
import string
import pandas as pd
from nltk.corpus import stopwords
import nltk

# download stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

# ================= TF-IDF =================
# lowercase + remove punctuation + remove stopword
def preprocess_tfidf(text):

    text = str(text).lower()

    # hapus punctuation
    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    # hapus whitespace berlebih
    text = re.sub(r"\s+", " ", text).strip()

    # stopword removal
    words = text.split()

    words = [
        word for word in words
        if word not in stop_words
    ]

    return " ".join(words)


# ================= EMBEDDING =================
# lowercase + remove punctuation
def preprocess_embed(text):

    text = str(text).lower()

    # hapus punctuation
    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    # hapus whitespace berlebih
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ================= EMBEDDING WITH PUNCTUATION =================
# lowercase only
def preprocess_embed_with_punctuation(text):

    text = str(text).lower()

    # rapikan whitespace aja
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ================= APPLY =================
# bikin kolom baru langsung

df["text_for_tfidf"] = df["translated_text"].apply(
    preprocess_tfidf
)

df["text_for_embed"] = df["translated_text"].apply(
    preprocess_embed
)

df["text_for_embed_with_punctuation"] = df["translated_text"].apply(
    preprocess_embed_with_punctuation
)


# ================= FINAL DATAFRAME =================
final_df = df[
    [
        "candidate_name",
        "email",
        "phone",
        "skills",
        "experience",
        "degree",
        "university",
        "Category",

        # preprocessing
        "text_for_tfidf",
        "text_for_embed",
        "text_for_embed_with_punctuation"
    ]
]


# ================= SAVE =================
final_df.to_csv("data_clean.csv", index=False)

print("✅ CSV berhasil diupdate")

print(final_df.head())

✅ CSV berhasil diupdate
   candidate_name                       email           phone  \
0  Siti Nurhaliza  siti.nurhaliza@example.com   6281234567890   
1      Rina Aulia      rina.aulia@example.com   6281234567890   
2    Anna Suryani    anna.suryani@example.com   6281234567890   
3    Andi Pratama    andi.pratama@example.com   6281123456789   
4        John Doe          john.doe@email.com  62898765432100   

                                              skills  \
0  penanganan keluhan, komunikasi efektif, penyel...   
1  komunikasi efektif, crm software, salesforce, ...   
2  problem-solving, intern midwife august 2023 - ...   
3  sistem pos, komunikasi efektif, manajemen inve...   
4  assisted in planning and executing marketing c...   

                                          experience degree  \
0  lebih dari 2 tahun dalam menangani pertanyaan ...    SMA   
1  3 tahun dalam memberikan pelayanan yang optima...     D3   
2  d in providing comprehensive care to expectant...     S1

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
import matplotlib.pyplot as plt

# Hitung top 10 lokasi
top_locations = df["Lokasi"].value_counts().head(10)

# Plot
plt.figure()
top_locations.plot(kind="bar")

plt.title("Top 10 Distribusi Lokasi")
plt.xlabel("Lokasi")
plt.ylabel("Jumlah")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
type_counts = df["Type"].value_counts()

plt.figure()
plt.pie(type_counts, labels=type_counts.index, autopct="%1.1f%%")
plt.title("Persebaran Kolom Type")
plt.tight_layout()
plt.show()

In [ ]:

# Buat kategori
gaji_status = df["Gaji"].apply(lambda x: "Ada Gaji" if x != "-" else "Tidak Ada")

# Hitung jumlah
gaji_counts = gaji_status.value_counts()

# Plot pie chart
plt.figure()
plt.pie(gaji_counts, labels=gaji_counts.index, autopct="%1.1f%%")

plt.title("Persebaran Data Gaji")

plt.show()

In [48]:
# cek hasil
final_df[["text_for_tfidf","text_for_embed","text_for_embed_with_punctuation"]].head()

,text_for_tfidf,text_for_embed,text_for_embed_with_punctuation
0,summary experience 2 years efficiently handlin...,summary experience more than 2 years in effici...,summary: experience: more than 2 years in effi...
1,summary experience 3 years providing optimal s...,summary experience 3 years in providing optima...,summary: experience: 3 years in providing opti...
2,summary experience providing comprehensive car...,summary experience d in providing comprehensiv...,summary: experience: d in providing comprehens...
3,summary experience 3 years coffee industry ski...,summary experience more than 3 years in the co...,summary: experience: more than 3 years in the ...
4,summary recent high school graduate strong aca...,summary a recent high school graduate with a s...,summary: a recent high school graduate with a ...


In [53]:
import re
import string
import pandas as pd
from nltk.corpus import stopwords
import nltk

# download stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

# ================= TF-IDF =================
# lowercase + hapus punctuation + stopword
def preprocess_tfidf(text):

    text = str(text).lower()

    # hapus punctuation
    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    # hapus whitespace berlebih
    text = re.sub(r"\s+", " ", text).strip()

    # stopword removal
    words = text.split()

    words = [
        word for word in words
        if word not in stop_words
    ]

    return " ".join(words)


# ================= EMBEDDING =================
# lowercase + hapus punctuation
def preprocess_embed(text):

    text = str(text).lower()

    # hapus punctuation
    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    # hapus whitespace berlebih
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ================= EMBEDDING WITH PUNCTUATION =================
# lowercase only
def preprocess_embed_with_punctuation(text):

    text = str(text).lower()

    # rapikan whitespace aja
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ================= APPLY =================
df["text_for_tfidf"] = df["translated_text"].apply(
    preprocess_tfidf
)

df["text_for_embed"] = df["translated_text"].apply(
    preprocess_embed
)

df["text_for_embed_with_punctuation"] = df["translated_text"].apply(
    preprocess_embed_with_punctuation
)


# ================= FINAL DATAFRAME =================
final_df = df[
    [
        "candidate_name",
        "email",
        "phone",
        "skills",
        "summary",
        "experience",
        "degree",
        "university",
        "Category",

        "text_for_tfidf",
        "text_for_embed",
        "text_for_embed_with_punctuation"
    ]
]


# ================= SAVE =================
final_df.to_csv("hasil scan cv parse.csv", index=False)

print("✅ CSV berhasil diupdate")

final_df.head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


✅ CSV berhasil diupdate


,candidate_name,email,phone,skills,summary,experience,degree,university,Category,text_for_tfidf,text_for_embed,text_for_embed_with_punctuation
0,Siti Nurhaliza,siti.nurhaliza@example.com,6281234567890,"penanganan keluhan, komunikasi efektif, penyel...",NONE,lebih dari 2 tahun dalam menangani pertanyaan ...,SMA,lorem,Hospitality,summary experience 2 years efficiently handlin...,summary experience more than 2 years in effici...,summary: experience: more than 2 years in effi...
1,Rina Aulia,rina.aulia@example.com,6281234567890,"komunikasi efektif, crm software, salesforce, ...",NONE,3 tahun dalam memberikan pelayanan yang optima...,D3,universitas lorem ipsum jakarta lulus dengan p...,Customer Service,summary experience 3 years providing optimal s...,summary experience 3 years in providing optima...,summary: experience: 3 years in providing opti...
2,Anna Suryani,anna.suryani@example.com,6281234567890,"problem-solving, intern midwife august 2023 - ...",NONE,d in providing comprehensive care to expectant...,S1,university of indonesia jakarta focused on mat...,Healthcare,summary experience providing comprehensive car...,summary experience d in providing comprehensiv...,summary: experience: d in providing comprehens...
3,Andi Pratama,andi.pratama@example.com,6281123456789,"sistem pos, komunikasi efektif, manajemen inve...",NONE,lebih dari 3 tahun di industri kopi. terampil ...,SMA,lorem,Customer Service,summary experience 3 years coffee industry ski...,summary experience more than 3 years in the co...,summary: experience: more than 3 years in the ...
4,John Doe,john.doe@email.com,62898765432100,assisted in planning and executing marketing c...,a recent high school graduate with a strong ac...,internship - marketing assistant august 2022 -...,D3,lorem,Marketing,summary recent high school graduate strong aca...,summary a recent high school graduate with a s...,summary: a recent high school graduate with a ...
